In [ ]:
!nvidia-smi
!git clone -q https://github.com/Amarnath10i/Research-On-Hyperspectral-ImageFussion.git /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q dc89aa8 && git log --oneline -1
!python -c "import torch, h5py; print(torch.__version__, torch.cuda.device_count(), torch.cuda.get_device_name(0))"
!find /kaggle/input -maxdepth 4 | head -30

In [ ]:
# 1) PUFormer training (checkpoints: last.pt every eval, best_ema.pt, EMA snapshots)
import subprocess, sys
p = subprocess.Popen("python train.py --mat /kaggle/input --cache /tmp/chikusei_crop.npy --model puformer --width 48 --stages 3 --hours 10.8 --bs 16 --lr 1.5e-4 --epochs 2000 --eval_epochs 40 --snap_epochs 200 --w_sam 0.05 --w_ssim 0.1 --init_ckpt $(find /kaggle/input -path '*init*' -name best_ema.pt | head -1) --out /kaggle/working/out/puformer", shell=True, cwd='/kaggle/working/repo/methods/puformer',
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end='', flush=True)
assert p.wait() == 0, 'command failed'


In [ ]:
import json, glob
for f in sorted(glob.glob('/kaggle/working/out/*/results.json')):
    r = json.load(open(f)); print(f, r['params_M'], r.get('iters'), r.get('epochs'))
    for k in ('test', 'test_tta'):
        if k in r: print(' ', k, {m: round(v, 4) for m, v in r[k].items()})
!rm -rf /kaggle/working/repo; ls -la /kaggle/working/out/*